# Revision analyses and figures — CTS-2026-0457-T

**Development of a European Union Time-Indexed Reference Dataset for Assessing the Performance of Signal Detection Methods in Pharmacovigilance using a Large Language Model**

This notebook reproduces **all analyses and figures introduced during peer review** (Reviewer #1 — Round 1; Reviewer #2 — Round 2) for the manuscript above. It is a self-contained addendum to `Software/manuscript_analysis.py`; the original analysis is unchanged.

| Section | Reviewer point | What it produces |
|---|---|---|
| 1 | setup | Load `Processed_database.xlsx`, shared helpers |
| 2 | R2.3 | **Figure 1** — Panel B adds the standardised per-product rate of post-approval AE additions |
| 3 | R1.5 | **Figure 2** — Panel B ordered by small-molecule frequency |
| 4 | R1.6 / R2.8 | **Figure 3** — Panel A full WHO ATC level-1 names; **Panel C renamed** (R2.6) |
| 5 | R1.7 / R2.8 | **Figure 4** — clarified axis labels |
| 6 | R1.4 | **Figure S1** — accrual-adjusted specificity over time (fixed window) |
| 7 | R2.9 | **MedDRA mapping reconciliation** — full accounting to 100% |
| 8 | R2.1 | **Data-source / metadata summary** |

All figures are written at 600 dpi. Run the cells top-to-bottom.


## 1. Setup

In [ ]:
# In Google Colab, upload Processed_database.xlsx to the session (or mount Drive) and set DATA_PATH.
!pip -q install pandas numpy matplotlib seaborn openpyxl "plotly==5.24.1" "kaleido==0.2.1"

In [ ]:
import os
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec, matplotlib.image as mpimg
import seaborn as sns, io
import plotly.express as px, plotly.io as pio

DATA_PATH = os.environ.get("DATA_PATH", "Processed_database.xlsx")  # e.g. /content/Processed_database.xlsx in Colab
OUT = "revision_figures"; os.makedirs(OUT, exist_ok=True)

df = pd.read_excel(DATA_PATH)
df["Date Added"] = pd.to_datetime(df["Date Added"], errors="coerce")
df["Year Added"] = df["Date Added"].dt.year
print("Loaded:", df.shape, "rows x cols")

# ---- shared helpers ----
SPEC_ORDER  = ["Unique (1)", "Rare (2-5)", "Common (6-50)", "Ubiquitous (>50)"]
SPEC_COLORS = ["#440154", "#31688e", "#35b779", "#fde725"]

def specificity_band(n):
    return "Unique (1)" if n == 1 else "Rare (2-5)" if n <= 5 else "Common (6-50)" if n <= 50 else "Ubiquitous (>50)"

def panel_label(ax, lab, x=-0.15, y=1.1, fs=16):
    ax.text(x, y, lab, transform=ax.transAxes, fontsize=fs, fontweight="bold", va="top", ha="right")

# INN-suffix proxy for drug mechanism (as in the original analysis)
BIOLOGIC_SUFFIXES = ["mab", "nib", "zumab", "ximab", "lumab", "tinib", "rafenib", "ciclib"]
df["Drug_Type"] = df["inn"].apply(
    lambda x: "Biologic/Targeted" if any(str(x).lower().endswith(k) for k in BIOLOGIC_SUFFIXES)
    else "Small Molecule" if pd.notna(x) else "Unknown")

## 2. Figure 1 — per-product rate of post-approval AE additions (Reviewer #2, R2.3)

Panel B now shows the **absolute** annual count of post-approval AE additions (bars) together with the **per-product rate** (line, secondary axis), obtained by dividing each year's additions by the number of products marketed by that year. The absolute count peaks around 2012, whereas the per-product rate peaks in the mid-2000s and declines thereafter — showing that the absolute peak is driven substantially by growth of the centrally authorised portfolio. Panels A, C and D are unchanged from the original Figure 1.

In [ ]:
df["Launch_Date"] = df.groupby("Brand_Name")["Date Added"].transform("min")
df["Drug_Age_at_AE"] = df["Date Added"].dt.year - df["Launch_Date"].dt.year
vintage = df[df["Drug_Age_at_AE"] > 0].copy()   # post-marketing additions

fig = plt.figure(figsize=(11.69, 8.27)); gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.45)

# Panel A — cumulative growth
ax1 = fig.add_subplot(gs[0, 0])
growth = df.dropna(subset=["Date Added"]).groupby(df["Date Added"].dt.year).size().cumsum()
growth.plot(kind="line", marker="o", color="#2c3e50", linewidth=1.5, markersize=3, ax=ax1)
ax1.set_ylabel("Cumulative Drug-AE Pairs", fontsize=10); ax1.set_xlabel("Year Added to SmPC", fontsize=10)
ax1.grid(True, linestyle="--", alpha=0.4); ax1.tick_params(labelsize=8); panel_label(ax1, "A")

# Panel B — absolute count (bars) + per-product rate (line)  [R2.3]
ax2 = fig.add_subplot(gs[0, 1])
yearly = vintage.groupby(vintage["Date Added"].dt.year).size()
first_year = df.groupby("Brand_Name")["Date Added"].min().dt.year
marketed = np.array([int((first_year <= y).sum()) for y in yearly.index])
rate = yearly.values / marketed
ax2.bar(yearly.index, yearly.values, color="firebrick", alpha=0.30)
ax2.set_xlabel("Year of label update", fontsize=10)
ax2.set_ylabel("AEs added to SmPC (count)", fontsize=10, color="firebrick")
ax2.tick_params(axis="y", labelcolor="firebrick", labelsize=8); ax2.tick_params(axis="x", labelsize=8)
ax2b = ax2.twinx()
ax2b.plot(yearly.index, rate, marker="o", color="#17486b", linewidth=1.8, markersize=3)
ax2b.set_ylabel("AEs added per marketed product", fontsize=10, color="#17486b")
ax2b.tick_params(axis="y", labelcolor="#17486b", labelsize=8); ax2b.set_ylim(0, None)
ax2.grid(True, linestyle="--", alpha=0.4); panel_label(ax2, "B")

# Panel C — regulatory procedure types
ax3 = fig.add_subplot(gs[1, 0])
proc = df["Union_register_procedure"].value_counts().head(10)
sns.barplot(x=proc.values, y=proc.index, palette="magma", ax=ax3)
ax3.set_xlabel("Number of Drug-AE Associations", fontsize=10); ax3.set_ylabel("")
ax3.tick_params(axis="y", labelsize=7); ax3.tick_params(axis="x", labelsize=8); panel_label(ax3, "C")

# Panel D — clean-label survival
ax4 = fig.add_subplot(gs[1, 1]); total = df["Brand_Name"].nunique()
first_update = vintage.groupby("Brand_Name")["Drug_Age_at_AE"].min().reset_index()
yrs = range(0, 21)
clean = [(1 - (first_update[first_update["Drug_Age_at_AE"] <= y]["Brand_Name"].nunique() / total)) * 100 for y in yrs]
ax4.plot(list(yrs), clean, marker="o", color="navy", markersize=4, linewidth=1.5)
ax4.set_xlabel("Years Post-Launch", fontsize=10); ax4.set_ylabel("% Drugs with No Post-Marketing Updates", fontsize=10)
ax4.set_ylim(0, 105); ax4.grid(True, linestyle="--", alpha=0.4); ax4.tick_params(labelsize=8); panel_label(ax4, "D")

plt.subplots_adjust(left=0.18, right=0.92, top=0.92, bottom=0.1)
plt.savefig(f"{OUT}/Figure1.png", dpi=600, bbox_inches="tight"); plt.show()

rate_table = pd.DataFrame({"Year": yearly.index, "PostMkt_AE_added": yearly.values,
                           "Products_marketed": marketed, "AE_per_product": np.round(rate, 3)})
print("Absolute peak:", int(yearly.idxmax()), int(yearly.max()),
      "| Rate peak:", int(yearly.index[int(np.argmax(rate))]), round(float(np.max(rate)), 2))
rate_table

## 3. Figure 2 — mechanism-based SOC profile, Panel B ordered (Reviewer #1, R1.5)

Panel A: introduction of new MedDRA Preferred Terms per year (bars) and cumulative unique PTs (line). Panel B: proportional distribution across the top-12 System Organ Classes for small molecules vs biologic/targeted therapies, with the y-axis **ordered from most to least frequent for small molecules** (R1.5).

In [ ]:
clean_df = df.dropna(subset=["MedDRA_PT_Term", "Year Added"]).copy()
pt_first = clean_df.groupby("MedDRA_PT_Term")["Year Added"].min().reset_index(name="First_Seen_Year")
nov = pt_first.groupby("First_Seen_Year").size().reset_index(name="New_PTs")
nov["Cumulative_PTs"] = nov["New_PTs"].cumsum()

fig1 = plt.figure(figsize=(11.69, 8.27)); gs1 = gridspec.GridSpec(2, 1, figure=fig1, hspace=0.3)
def pl2(ax, lab): ax.text(-0.05, 1.1, lab, transform=ax.transAxes, fontsize=18, fontweight="bold", va="top")

ax1a = fig1.add_subplot(gs1[0, 0]); ax1b = ax1a.twinx()
ax1a.bar(nov["First_Seen_Year"], nov["New_PTs"], color="steelblue", alpha=0.5)
ax1b.plot(nov["First_Seen_Year"], nov["Cumulative_PTs"], color="firebrick", linewidth=2, marker="o", markersize=4)
ax1a.set_ylabel("New MedDRA PTs Introduced", color="steelblue", fontsize=12, fontweight="bold")
ax1b.set_ylabel("Cumulative Unique PTs", color="firebrick", fontsize=12, fontweight="bold")
ax1a.set_xlabel("Year of Introduction", fontsize=12); pl2(ax1a, "A")

ax2 = fig1.add_subplot(gs1[1, 0])
top_socs = df["MedDRA_SOC_Term"].value_counts().head(12).index
soc_type = df[df["MedDRA_SOC_Term"].isin(top_socs) & df["Drug_Type"].isin(["Biologic/Targeted", "Small Molecule"])]
soc_type = soc_type.groupby(["Drug_Type", "MedDRA_SOC_Term"]).size().reset_index(name="Count")
soc_type["Pct"] = soc_type.groupby("Drug_Type")["Count"].transform(lambda x: x / x.sum() * 100)
sm_order = soc_type[soc_type["Drug_Type"] == "Small Molecule"].sort_values("Pct", ascending=False)["MedDRA_SOC_Term"].tolist()
for s in top_socs:
    if s not in sm_order: sm_order.append(s)
sns.barplot(data=soc_type, x="Pct", y="MedDRA_SOC_Term", hue="Drug_Type", order=sm_order, palette="Set1", ax=ax2)
ax2.set_xlabel("Percentage of Total AEs within Drug Type (%)", fontsize=12, fontweight="bold")
ax2.set_ylabel(""); ax2.legend(fontsize=10, title="Drug Type", title_fontsize=11, loc="center right"); pl2(ax2, "B")

plt.subplots_adjust(left=0.2, right=0.9, top=0.9, bottom=0.1)
plt.savefig(f"{OUT}/Figure2.png", dpi=600, bbox_inches="tight"); plt.show()

## 4. Figure 3 — ATC names (R1.6) and renamed Panel C (R2.6)

Panel A shows each WHO ATC level-1 code with its **full anatomical main-group name** (R1.6). **Panel C has been renamed and relabelled** (R2.6): title *"Cross-product sharing of adverse-event terms"*, x-axis *"Number of distinct products in which the PT appears"* (bands: Unique = 1, Rare = 2–5, Common = 6–50, Ubiquitous = >50), y-axis *"Number of distinct MedDRA PTs"*; the term "uniquity" has been removed. Panel B (pre/post by SOC) and Panel D (ATC sunburst) are unchanged.

In [ ]:
pt_counts = df.groupby("MedDRA_PT_Term")["Brand_Name"].nunique().reset_index(name="Drug_Count")
pt_counts["Category"] = pt_counts["Drug_Count"].apply(specificity_band)

atc_hier = df.dropna(subset=["ATC_Level_1_Desc", "ATC_Level_2_Desc"]).copy()
ach = atc_hier.groupby(["ATC_Level_1_Desc", "ATC_Level_2_Desc"]).size().reset_index(name="Count")
fs = px.sunburst(ach, path=["ATC_Level_1_Desc", "ATC_Level_2_Desc"], values="Count",
                 color="Count", color_continuous_scale="RdBu_r")
fs.update_layout(margin=dict(t=10, l=10, r=10, b=10))
sun = mpimg.imread(io.BytesIO(pio.to_image(fs, format="png", scale=4)))

fig = plt.figure(figsize=(18.39, 12.27)); gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.55)

# Panel A — full ATC level-1 names
ax1 = fig.add_subplot(gs[0, 0])
code2desc = (df.dropna(subset=["ATC_Level_1_Code", "ATC_Level_1_Desc"]).drop_duplicates("ATC_Level_1_Code")
             .set_index("ATC_Level_1_Code")["ATC_Level_1_Desc"].to_dict())
atc = df["ATC_Level_1_Code"].value_counts().head(15)
ylabels = [f"{c} – {code2desc.get(c,'')}" for c in atc.index]
sns.barplot(x=atc.values, y=ylabels, palette="viridis", ax=ax1)
ax1.set_xlabel("Number of Drug-AE Pairs", fontsize=11); ax1.set_ylabel("WHO ATC level 1 anatomical main group", fontsize=11)
ax1.tick_params(labelsize=9); panel_label(ax1, "A")

# Panel B — pre/post by SOC
ax2 = fig.add_subplot(gs[0, 1])
uvs = df.groupby(["MedDRA_SOC_Term", "Source"]).size().unstack().fillna(0); uvs["T"] = uvs.sum(axis=1)
uvs = uvs.sort_values("T", ascending=False).head(15).drop("T", axis=1)
uvs.plot(kind="barh", stacked=True, color=["#2a9d8f", "#e76f51"], ax=ax2)
ax2.set_xlabel("Number of Drug-AE Pairs", fontsize=10); ax2.set_ylabel(""); ax2.tick_params(axis="y", labelsize=7)
ax2.legend(title="Discovery Phase", fontsize=8, title_fontsize=9, loc="upper right", frameon=True); panel_label(ax2, "B")

# Panel C — RENAMED (R2.6)
ax3 = fig.add_subplot(gs[1, 0])
sns.countplot(data=pt_counts, x="Category", order=SPEC_ORDER, palette="viridis", ax=ax3)
ax3.set_xlabel("Number of distinct products in which the PT appears", fontsize=11)
ax3.set_ylabel("Number of distinct MedDRA PTs", fontsize=11)
ax3.set_title("Cross-product sharing of adverse-event terms", fontsize=12, fontweight="bold")
tot = len(pt_counts)
for p in ax3.patches:
    ax3.annotate("{:.1f}%".format(100 * p.get_height() / tot),
                 (p.get_x() + p.get_width() / 2, p.get_height() + 20), ha="center", fontsize=9, fontweight="bold")
panel_label(ax3, "C")

# Panel D — ATC sunburst
ax4 = fig.add_subplot(gs[1, 1]); ax4.imshow(sun); ax4.axis("off"); panel_label(ax4, "D")

plt.subplots_adjust(left=0.2, right=0.95, top=0.9, bottom=0.1)
plt.savefig(f"{OUT}/Figure3.png", dpi=600, bbox_inches="tight"); plt.show()

## 5. Figure 4 — clarified axis labels (Reviewer #1 R1.7; Reviewer #2 R2.8)

Panel A: per-drug burden of unique MedDRA PTs — histogram (x = *Unique MedDRA PTs per Drug*, y = *Number of Drugs*) and boxplot by ATC level-1. Panel B: multi-system toxicity — distinct SOCs per drug, with the same explicit labelling. The ambiguous "count" axis has been named.

In [ ]:
ae_per_drug = df.groupby("Brand_Name")["MedDRA_PT_Term"].nunique().reset_index(name="AE_Count")
ae_per_drug = ae_per_drug.merge(df[["Brand_Name", "ATC_Level_1_Desc"]].drop_duplicates(), on="Brand_Name")
soc_per_drug = df.groupby("Brand_Name")["MedDRA_SOC_Term"].nunique().reset_index(name="SOC_Count")
soc_per_drug = soc_per_drug.merge(df[["Brand_Name", "ATC_Level_1_Desc"]].drop_duplicates(), on="Brand_Name")

fig2 = plt.figure(figsize=(11.69, 8.27)); gs2 = gridspec.GridSpec(2, 1, figure=fig2, hspace=0.4)
def pl3(ax, lab): ax.text(-0.05, 1.1, lab, transform=ax.transAxes, fontsize=18, fontweight="bold", va="top")

ia = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs2[0, 0], wspace=0.4, width_ratios=[1, 1.4])
ax3h = fig2.add_subplot(ia[0, 0]); ax3b = fig2.add_subplot(ia[0, 1])
sns.histplot(ae_per_drug["AE_Count"], bins=30, kde=True, ax=ax3h, color="steelblue")
ax3h.axvline(ae_per_drug["AE_Count"].median(), color="red", linestyle="--", linewidth=2)
ax3h.set_xlabel("Unique MedDRA PTs per Drug", fontsize=11, fontweight="bold")
ax3h.set_ylabel("Number of Drugs", fontsize=11, fontweight="bold"); ax3h.tick_params(labelsize=10)
order_c = ae_per_drug.groupby("ATC_Level_1_Desc")["AE_Count"].median().sort_values().index
sns.boxplot(data=ae_per_drug, y="ATC_Level_1_Desc", x="AE_Count", order=order_c, palette="Set2", ax=ax3b)
ax3b.yaxis.tick_right(); ax3b.set_xlabel("Unique MedDRA PTs per Drug", fontsize=11, fontweight="bold")
ax3b.set_ylabel(""); ax3b.tick_params(labelsize=9); pl3(ax3h, "A")

ib = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs2[1, 0], wspace=0.4, width_ratios=[1, 1.4])
ax4h = fig2.add_subplot(ib[0, 0]); ax4b = fig2.add_subplot(ib[0, 1])
sns.histplot(soc_per_drug["SOC_Count"], bins=range(1, 28), discrete=True, ax=ax4h, color="purple")
ax4h.axvline(soc_per_drug["SOC_Count"].median(), color="red", linestyle="--", linewidth=2)
ax4h.set_xlabel("Distinct SOCs per Drug", fontsize=11, fontweight="bold")
ax4h.set_ylabel("Number of Drugs", fontsize=11, fontweight="bold"); ax4h.tick_params(labelsize=10)
order_d = soc_per_drug.groupby("ATC_Level_1_Desc")["SOC_Count"].median().sort_values().index
sns.boxplot(data=soc_per_drug, y="ATC_Level_1_Desc", x="SOC_Count", order=order_d, palette="Purples", ax=ax4b)
ax4b.yaxis.tick_right(); ax4b.set_xlabel("Distinct SOCs per Drug", fontsize=11, fontweight="bold")
ax4b.set_ylabel(""); ax4b.tick_params(labelsize=9); pl3(ax4h, "B")

plt.subplots_adjust(left=0.1, right=0.82, top=0.9, bottom=0.1)
plt.savefig(f"{OUT}/Figure4.png", dpi=600, bbox_inches="tight"); plt.show()

## 6. Figure S1 — accrual-adjusted specificity over time (Reviewer #1, R1.4)

Whether the balance between product-specific and ubiquitous terms shifted over time is confounded by **right-censoring**: a PT first recorded near the data-lock has had little time to spread to other products. To remove this, each PT is assigned to its year of first appearance and classified by the number of products listing it **within a fixed K-year window** (default K = 3), so every annual cohort has an identical observation period; cohorts without a full window (after `max_year − K`) are excluded. Once accrual time is held constant the composition is comparatively stable across the three decades, and no term reaches >50 products within K years — i.e. the apparent drift toward uniqueness in the naive view is largely an accrual artefact.

In [ ]:
K = 3  # accrual window in years; set to 5 for a stricter window
maxy = int(df["Year Added"].max()); last_cohort = maxy - K
valid = df.dropna(subset=["MedDRA_PT_Term", "Year Added"]).copy()

pt_product_year = valid.groupby(["MedDRA_PT_Term", "Brand_Name"])["Year Added"].min().reset_index(name="product_year")
pt_first_year  = valid.groupby("MedDRA_PT_Term")["Year Added"].min().reset_index(name="first_year")
m = pt_product_year.merge(pt_first_year, on="MedDRA_PT_Term")
m = m[m["product_year"] <= m["first_year"] + K]
win = (m.groupby("MedDRA_PT_Term")["Brand_Name"].nunique().reset_index(name="wc")
         .merge(pt_first_year, on="MedDRA_PT_Term"))
win["first_year"] = win["first_year"].astype(int)
win["Category"] = win["wc"].apply(specificity_band)
cohort = win[(win["first_year"] >= 1995) & (win["first_year"] <= last_cohort)]

comp = (cohort.groupby(["first_year", "Category"]).size().unstack(fill_value=0)
              .reindex(columns=SPEC_ORDER, fill_value=0))
prop = comp.div(comp.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11.69, 6.5)); bottom = np.zeros(len(prop))
for c, col in zip(SPEC_ORDER, SPEC_COLORS):
    ax.bar(prop.index, prop[c], bottom=bottom, label=c, color=col, width=0.85); bottom += prop[c].values
ax.set_xlabel("Year of first appearance of the MedDRA PT (cohort)", fontsize=12)
ax.set_ylabel("Proportion of newly introduced PTs (%)", fontsize=12)
ax.set_ylim(0, 100); ax.set_xlim(prop.index.min() - 0.5, prop.index.max() + 0.5)
ax.set_title(f"Accrual-adjusted specificity of newly introduced MedDRA PTs (fixed {K}-year window)",
             fontsize=12, fontweight="bold")
ax.legend(title=f"Products sharing the PT within {K} years of first appearance",
          fontsize=9, title_fontsize=10, ncol=4, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout(); plt.savefig(f"{OUT}/Figure_S1_specificity_over_time.png", dpi=600, bbox_inches="tight"); plt.show()
print(f"Cohorts {int(prop.index.min())}-{int(prop.index.max())}; PTs={len(cohort)}; "
      f">50 within {K}y = {int((cohort['Category']=='Ubiquitous (>50)').sum())}")

## 7. MedDRA mapping reconciliation (Reviewer #2, R2.9)

The reviewer noted that the reported mapping figures did not sum to 100%. The full accounting over all associations is below: automated mapping (exact + SOC-filtered) + manual assignment after no automated match + processing errors + associations retained without a MedDRA PT (excluded from PT-level analyses).

In [ ]:
vc = df["MedDRA_Match_Method"].value_counts(dropna=False)
tot = len(df)
tab = (vc.rename_axis("Match method").reset_index(name="n"))
tab["%"] = (100 * tab["n"] / tot).round(3)
tab["Match method"] = tab["Match method"].fillna("No MedDRA PT assigned (unmapped)")
auto = 100 * df["MedDRA_Match_Method"].isin(["Exact Match", "SOC-Filtered Batch Match"]).mean()
print(f"Total associations: {tot:,}")
print(f"Automated mapping (Exact + SOC-filtered): {auto:.1f}%")
print(f"With a MedDRA PT assigned: {100*df['MedDRA_PT_Term'].notna().mean():.1f}%")
tab

## 8. Data sources and metadata (Reviewer #2, R2.1)

Two extraction operations: (i) LLM-based AE extraction from SmPC Section 4.8; (ii) rule-based parsing of regulatory/pharmacological metadata from the Union Register HTML/JSON — complemented by MedDRA v28 and WHO ATC dictionary mapping. The processed database carries 36 variables per drug–AE association; the cell below lists them grouped by origin.

In [ ]:
groups = {
 "Drug identification (Union Register)": ["Brand_Name","inn","Union_register_eu_num","Union_register_mah"],
 "Adverse-event information (SmPC Section 4.8 / LLM)": ["LLM_extracted_AE","Source","Reference Date","Date Added"],
 "MedDRA v28 classification": ["MedDRA_PT_Term","MedDRA_PT_Code","MedDRA_HLT_Term","MedDRA_HLT_Code",
     "MedDRA_HLGT_Term","MedDRA_HLGT_Code","MedDRA_SOC_Term","MedDRA_SOC_Code","MedDRA_Match_Method"],
 "WHO ATC classification": [c for c in df.columns if c.startswith("ATC_Level_")],
 "Regulatory metadata (Union Register)": [c for c in df.columns if c.startswith("Union_register_")],
 "Traceability": ["Source_File"],
}
for g, cols in groups.items():
    present = [c for c in cols if c in df.columns]
    print(f"\n{g}  ({len(present)} fields)"); print("  " + ", ".join(present))
print(f"\nTotal columns in processed database: {df.shape[1]}")

## 9. Example of the Union Register HTML source and its parsing (Reviewer #2, R2.1)

The regulatory and pharmacological metadata are **not scraped from rendered tables**: each Union Register product page embeds its data as JavaScript variables in the HTML source, which the pipeline parses directly. The screenshot `union_register_source_example.png` (in this folder) shows the `view-source:` of a product page; the arrays below are reproduced (trimmed) from that page.

- Example product: **Kineret** (anakinra)
- URL: `https://ec.europa.eu/health/documents/community-register/html/h203.htm`
- `var dataSet_product_information = [...]` → product-level metadata (name, EU number, INN, MAH, full ATC hierarchy, EMA links)
- `var dataSet_proc = [...]` → one row per regulatory procedure; the `closed` date is the **temporal anchor** used to time-index each drug–AE pair.

| Source field | Processed-database variable(s) |
|---|---|
| `name.value` | `Brand_Name` |
| `eu_num.value` | `Union_register_eu_num` |
| `inn.value` | `inn` |
| `mah.value` (+ `meta.address`) | `Union_register_mah` |
| `atc.meta[0]` (levels 1–5) | `ATC_Level_1..5_Code` / `_Desc`, `Union_register_atc` |
| `ema_links.meta[].url` | `Union_register_link` |
| `proc[].closed` | `Union_register_close_date` → **Date Added / Reference Date** |
| `proc[].type` | `Union_register_procedure` |
| `proc[].ema_number` | `Union_register_Ema_number` |
| `proc[].decision.number` / `.date` | `Union_register_decisio_number` / `Union_register_decision_date` |

The cell below parses a trimmed, offline copy of these arrays to show the mechanism (no network access required).

In [ ]:
import json

# Trimmed, offline copy of the two embedded arrays from h203.htm (Kineret).
SAMPLE_PRODUCT = '''[
 {"type":"name","value":"Kineret","meta":{"id":"203","status_name":"ACTIVE"}},
 {"type":"eu_num","value":"EU/1/02/203","meta":{}},
 {"type":"inn","value":"Anakinra","meta":{}},
 {"type":"mah","value":"Swedish Orphan Biovitrum AB (publ)","meta":{"id":"17479","address":"112 76 Stockholm, Sverige"}},
 {"type":"atc","value":null,"meta":[[
     {"level":"1","code":"L","description":"Antineoplastic and immunomodulating agents"},
     {"level":"2","code":"L04","description":"Immunosuppressants"},
     {"level":"3","code":"L04A","description":"Immunosuppressants"},
     {"level":"4","code":"L04AC","description":"Interleukin inhibitors"}]]},
 {"type":"ema_links","value":null,"meta":[{"url":"https://www.ema.europa.eu/en/medicines/human/EPAR/kineret","description":"EMA - Kineret"}]}
]'''
SAMPLE_PROC = '''[
 {"id":"4978","closed":"2002-03-12","type":"Centralised - Authorisation","ema_number":"EMEA/H/C/363","decision":{"number":"(2002) 1045","date":"2002-03-08"}},
 {"id":"5560","closed":"2002-07-15","type":"Centralised - Variation","ema_number":"EMEA/H/C/363/I/1","decision":{"number":"(2002) 2732","date":"2002-07-11"}}
]'''

prod = {d["type"]: d for d in json.loads(SAMPLE_PRODUCT)}
record = {
    "Brand_Name":            prod["name"]["value"],
    "Union_register_eu_num": prod["eu_num"]["value"],
    "inn":                   prod["inn"]["value"],
    "Union_register_mah":    prod["mah"]["value"],
    "Union_register_link":   prod["ema_links"]["meta"][0]["url"],
}
for lv in prod["atc"]["meta"][0]:
    record[f"ATC_Level_{lv['level']}_Code"] = lv["code"]
    record[f"ATC_Level_{lv['level']}_Desc"] = lv["description"]

print("Product-level metadata parsed from dataSet_product_information:")
for k, v in record.items():
    print(f"  {k:24s} = {v}")

proc = json.loads(SAMPLE_PROC)
procedures = pd.DataFrame([{
    "Union_register_close_date":   p["closed"],          # temporal anchor
    "Union_register_procedure":    p["type"],
    "Union_register_Ema_number":   p["ema_number"],
    "Union_register_decisio_number": p["decision"]["number"],
    "Union_register_decision_date":  p["decision"]["date"],
} for p in proc])
print("\nRegulatory-procedure metadata parsed from dataSet_proc:")
procedures